In [0]:
spark.conf.set(
    "fs.azure.account.key.strcm.blob.core.windows.net",
    dbutils.secrets.get("databricksScope", "secretkv")
)

In [0]:
dbutils.fs.ls("wasbs://raw-data@strcm.blob.core.windows.net/")

[FileInfo(path='wasbs://raw-data@strcm.blob.core.windows.net/accounts.csv', name='accounts.csv', size=4670, modificationTime=1780060904000),
 FileInfo(path='wasbs://raw-data@strcm.blob.core.windows.net/data_dictionary.csv', name='data_dictionary.csv', size=996, modificationTime=1780060920000),
 FileInfo(path='wasbs://raw-data@strcm.blob.core.windows.net/products.csv', name='products.csv', size=171, modificationTime=1780060940000),
 FileInfo(path='wasbs://raw-data@strcm.blob.core.windows.net/sales_pipeline.csv', name='sales_pipeline.csv', size=637773, modificationTime=1780060953000),
 FileInfo(path='wasbs://raw-data@strcm.blob.core.windows.net/sales_teams.csv', name='sales_teams.csv', size=1284, modificationTime=1780060970000)]

In [0]:
accounts_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://raw-data@strcm.blob.core.windows.net/accounts.csv")

In [0]:
data_dictionary_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://raw-data@strcm.blob.core.windows.net/data_dictionary.csv")

In [0]:
products_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://raw-data@strcm.blob.core.windows.net/products.csv")

In [0]:
sales_pipeline_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://raw-data@strcm.blob.core.windows.net/sales_pipeline.csv")

In [0]:
sales_teams_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://raw-data@strcm.blob.core.windows.net/sales_teams.csv")

In [0]:
print(accounts_df.columns)
print(data_dictionary_df.columns)
print(products_df.columns)
print(sales_pipeline_df.columns)
print(sales_teams_df.columns)

['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']
['Table', 'Field', 'Description']
['product', 'series', 'sales_price']
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
['sales_agent', 'manager', 'regional_office']


In [0]:
accounts_df = accounts_df.withColumnRenamed("subsidiary_of", "parent_company")
data_dictionary_df = data_dictionary_df.withColumnRenamed("Table", "table").withColumnRenamed("Field", "field").withColumnRenamed("Description", "description")

In [0]:
from pyspark.sql.functions import col, when, sum

null_counts_accounts_df = accounts_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in accounts_df.columns
])

display(null_counts_accounts_df)

account,sector,year_established,revenue,employees,office_location,parent_company
0,0,0,0,0,0,0


In [0]:
null_counts_data_dictionary_df = data_dictionary_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in data_dictionary_df.columns
])

display(null_counts_data_dictionary_df)

table,field,description
0,0,0


In [0]:
null_counts_products_df = products_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in products_df.columns
])

display(null_counts_products_df)

product,series,sales_price
0,0,0


In [0]:
null_counts_sales_pipeline_df = sales_pipeline_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in sales_pipeline_df.columns
])

display(null_counts_sales_pipeline_df)

opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,0,0,0,0,500,2089,2089


In [0]:
null_counts_sales_teams_df = sales_teams_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in sales_teams_df.columns
])

display(null_counts_sales_teams_df)

sales_agent,manager,regional_office
0,0,0


In [0]:
accounts_df = accounts_df.fillna({
    "parent_company" : "Independent"
})

sales_pipeline_df = sales_pipeline_df.fillna({
    "account" : "unknown"
})

In [0]:
accounts_df.write.option("header", "true").csv("wasbs://transformed-data@strcm.blob.core.windows.net/accounts.csv")

In [0]:
data_dictionary_df.write.option("header", "true").csv("wasbs://transformed-data@strcm.blob.core.windows.net/data_dictionary.csv")

In [0]:
products_df.write.option("header", "true").csv("wasbs://transformed-data@strcm.blob.core.windows.net/products.csv")

In [0]:
sales_pipeline_df.write.option("header", "true").csv("wasbs://transformed-data@strcm.blob.core.windows.net/sales_pipeline.csv")

In [0]:
sales_teams_df.write.option("header", "true").csv("wasbs://transformed-data@strcm.blob.core.windows.net/sales_teams.csv")